[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sumilee-pcu/llm-textbook/blob/main/chapters/part2_rag/ch05_rag.ipynb)

위 배지를 누르면 이 노트북이 구글 코랩에서 바로 열립니다.

# 5장 RAG: 검색 증강 생성
**「LLM 애플리케이션 입문 — RAG에서 멀티에이전트까지」 실습 노트북**

> 제2부 RAG — 검색으로 지식을 더하다
>
> Tier · `[T1]` 코랩 즉시 실행 · `[T2]` GPU/장시간 · `[T3]` 이론(개념 점검)
>
> 실습 코드 저장소: github.com/sumilee-pcu/llm-textbook

## 환경 설정 · 한글 폰트와 라이브러리
코랩에서 처음 한 번만 실행합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# 한글 폰트 설치 + matplotlib 한글 적용 (코랩 최초 1회)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.__init__()
for f in fm.fontManager.ttflist:
    if "NanumGothic" in f.name:
        plt.rcParams["font.family"] = fm.FontProperties(fname=f.fname).get_name(); break
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
# 실습 라이브러리 설치 (검증 시점 2026-06, 하한 고정)
!pip install -q "google-genai>=1.0" "langchain>=0.3" "langchain-google-genai>=1.0" "langchain-community>=0.3" "faiss-cpu>=1.7" "langchain-text-splitters>=0.3"

In [ ]:
# 설치된 핵심 라이브러리 버전 기록 (재현용) — 이 출력의 버전을 그대로 고정하면 가장 안정적이다
import importlib.metadata as _m
for _p in ["google-genai","langchain","langchain-google-genai","langchain-community",
           "langchain-text-splitters","langgraph","faiss-cpu","tiktoken","rank-bm25",
           "networkx","tavily-python"]:
    try: print(f"{_p}=={_m.version(_p)}")
    except Exception: pass

### API 키·모델·임베딩 설정

In [ ]:
# API 키 — 코랩 보안 비밀(시크릿)에 GOOGLE_API_KEY 등록 후 사용
#   왼쪽 열쇠 아이콘 > 새 보안 비밀 > 이름 GOOGLE_API_KEY > 값에 키 붙여넣기 > 노트북 액세스 ON
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
GEMINI_FLASH = "gemini-3-flash-preview"   # 모델 갱신 시 이 한 줄만 변경
EMBED_MODEL = "gemini-embedding-001"

### 4장의 embed·cosine 재사용

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

def embed(text):
    res = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return res.embeddings[0].values

## 예제 5-1. 겹침을 둔 텍스트 분할

In [ ]:
def split_text(text, size=200, overlap=40):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap     # 겹침만큼 덜 이동
    return chunks

sample = "연차는 입사 1년 후 15일이 주어진다. " * 20
parts = split_text(sample)
print("조각 수:", len(parts))
print("1번 조각 끝:", parts[0][-15:])
print("2번 조각 처음:", parts[1][:15])

## 예제 5-2. 프레임워크 없이 만드는 RAG

In [ ]:
docs = [
    "환불은 구매 후 7일 이내에 가능하다",
    "연차는 입사 1년 후 15일이 주어진다",
    "재택근무는 주 2회까지 가능하다",
]
doc_vecs = [embed(d) for d in docs]   # 4장의 embed, cosine 재사용

def retrieve(query, k=2):
    q = embed(query)
    ranked = sorted(zip(docs, doc_vecs), key=lambda x: cosine(q, x[1]), reverse=True)
    return [d for d, _ in ranked[:k]]

def rag_answer(query):
    context = "\n".join(f"- {d}" for d in retrieve(query))
    prompt = (
        f"다음 근거에만 기대어 답하고, 근거에 없으면 모른다고 답하라.\n"
        f"[근거]\n{context}\n[질문] {query}"
    )
    return client.models.generate_content(model=GEMINI_FLASH, contents=prompt).text

print(rag_answer("환불 언제까지 돼?"))
print(rag_answer("출장비는 얼마야?"))

## 예제 5-3. LangChain으로 RAG 구축

In [ ]:
# LangChain으로 RAG 구축 (4단계: 적재·분할·색인·검색생성)
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

raw = ["환불은 구매 후 7일 이내에 가능하다.",
       "연차는 입사 1년 후 15일이 주어진다.",
       "재택근무는 주 2회까지 가능하다."]
chunks = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40).create_documents(raw)

emb = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=API_KEY)
retriever = FAISS.from_documents(chunks, emb).as_retriever(search_kwargs={"k": 2})
llm = ChatGoogleGenerativeAI(model=GEMINI_FLASH, google_api_key=API_KEY, temperature=0)
prompt = ChatPromptTemplate.from_template(
    "다음 근거에만 기대어 답하고, 근거에 없으면 모른다고 답하라.\n[근거]\n{context}\n[질문] {question}")
chain = ({"context": retriever, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())
print(chain.invoke("환불 언제까지 돼?"))

## 심화 실습 `[T1]`
본문의 심화 실습 과제를 코랩에서 직접 구현해 본다. 책의 해당 장 끝 안내를 따른다.

## 정리
- 이 장의 예제를 모두 실행해 결과를 확인했다.
- 코드는 책 본문의 핵심을 옮긴 것이며, 확장 과제는 저장소 README를 참고한다.

> 저장소: github.com/sumilee-pcu/llm-textbook · 5장 노트북